# A quantum walk, decohered — watching $t$ turn into $\sqrt{t}$

**The punchline.** A walker that steps left or right according to a coin spreads out at a
rate that depends on *nothing but how much the rest of the world knows about the coin*.
Let the world learn nothing and the walker's width grows like $t$; let it learn everything
and the width grows like $\sqrt{t}$. Same walk, same gates, same number of steps — one
knob, turned continuously between two exponents that every textbook treats as belonging to
two different subjects.

The $\sqrt{t}$ end is not an approximation here. At full dephasing this circuit *is* a
simple random walk, and we assert $\sigma^2(t) = t$ to $10^{-9}$ and the whole position
distribution against the exact binomial to $10^{-15}$.

Three facts meet in one figure:

- **The Khintchine/CLT fact.** $t$ random $\pm 1$ steps add *in quadrature* — variance
  adds, so $\sigma \sim \sqrt{t}$. $t$ coherent $\pm 1$ steps add *linearly as amplitudes*
  and only then get squared, so $\sigma \sim t$. Same walk; different norm.
- **The transport gap.** Reaching distance $N$ takes $\sim N$ ballistic steps and
  $\sim N^2$ diffusive ones. That gap is the raw material of quantum-walk search.
- **The decoherence dial.** The exponent slides between the two as the coupling strength
  $\theta$ runs from $0$ to $\pi$ — and at $\theta = \pi$ the dephased Hadamard coin is
  *literally* the `COIN` matrix of
  **[l1_vs_l2](l1_vs_l2.ipynb)** Part 5, $\lvert H_{ij}\rvert^2$ row by row.

That last point is the reason this notebook exists next to E0. E0 shows the classical
theory sitting inside the quantum one *by construction* — dephase after every gate and the
simulator becomes `csim`. Here the same limit **emerges dynamically**: nobody imposes
classical dynamics, a coin gets watched, and simple random walk falls out with the
binomial coefficients correct to machine precision.

### The route

- **Part 1** builds the walk: a position register that counts like a binary odometer, a
  coin qubit, and a shift that is one block and its adjoint.
- **Part 2** runs it at seven coupling strengths and plots the final position profile —
  the famous two-horned ballistic shape dissolving into a Gaussian.
- **Part 3** is the money figure: $\sigma(t)$ on log–log axes, and the fitted exponent
  sliding from $1$ to $\tfrac12$.
- **Part 4** proves the $\theta = \pi$ endpoint exactly.
- **Part 5** is the honesty note: what the sliding exponent does *not* mean.
- **Part 6** two asides — why the coin is prepared the way it is, and what the coin's own
  coherence does along the way.

Background from the course: **[04 — combinators](../04-combinators.ipynb)** for blocks,
`control` and `within`; **[06 — why the world looks classical](../06-decoherence.ipynb)**
for `environment()` and `dephasing_coupling`. Companion exhibits:
**[decoherence_dial](decoherence_dial.ipynb)** (the knob on a single qubit),
**[quantum_eraser](quantum_eraser.ipynb)** (nothing here is lost), and
**[l1_vs_l2](l1_vs_l2.ipynb)** (the two norms).

In [ ]:
from math import comb

import matplotlib.pyplot as plt
import numpy as np

import qsim
from qsim import Circuit, Register
from qsim.decoherence import dephasing_coupling
from qsim.gates import H, S, X

N_POS = 6                                  # position qubits, so 2**6 = 64 sites
N_SITES = 2**N_POS
CENTER = N_SITES // 2                      # the walker starts here
SITES = np.arange(N_SITES) - CENTER        # signed position x of each site: -32 .. 31
T_STEPS = 12                               # one fresh environment qubit per step
STEPS = np.arange(1, T_STEPS + 1)
FIT_FROM = 3                               # fit the exponent over t in [3, T_STEPS]
THETAS = np.array([0.0, np.pi / 8, np.pi / 4, 3 * np.pi / 8,
                   np.pi / 2, 3 * np.pi / 4, np.pi])
THETA_LABELS = [r"$0$", r"$\pi/8$", r"$\pi/4$", r"$3\pi/8$",
                r"$\pi/2$", r"$3\pi/4$", r"$\pi$"]

np.set_printoptions(precision=4, suppress=True)

Nothing in this notebook is ever measured, so nothing in it is random: every number below
is a deterministic function of the state tensor. The `seed` passed to each `Circuit` is
there only because a demo with an unseeded generator is a demo that can surprise you later.

## Part 1 — Building the walk

A **coined quantum walk** is the quantum version of "flip a coin, step that way, repeat".
It needs two pieces:

- a **position register**: $n$ qubits holding a site number $0 \ldots 2^n - 1$ on a cycle
  (site $2^n - 1$ is adjacent to site $0$ — the line is bent into a ring so that "add one"
  is always defined);
- a **coin qubit**: one qubit whose two basis states mean *step right* and *step left*.

One step is three moves:

1. **Toss** — a Hadamard on the coin. Not a random choice: $H$ puts the coin into a
   superposition of both faces at once, which is what will let the two directions
   *interfere* later.
2. **Dephase** — let one fresh environment qubit look at the coin, by angle $\theta$. At
   $\theta = 0$ it learns nothing, at $\theta = \pi$ it learns the coin exactly.
3. **Shift** — add $1$ to the position where the coin is $\lvert 1\rangle$, subtract $1$
   where it is $\lvert 0\rangle$. Because the coin is in superposition, so is the walker's
   position afterwards; because the coin is *entangled with the position it produced*,
   the two branches remember which way they came.

That is the whole algorithm. The interesting part is entirely in step 2.

### 1a. Position as a binary odometer

The position register has to support "add 1, modulo $2^n$". On an MSB-first register
(`reg[0]` is the most significant bit — the convention everywhere in `qsim`) that is
exactly the mechanics of a car odometer:

> A digit rolls over **if and only if every digit below it is already at its maximum.**

In binary "at its maximum" means "is 1", and "rolls over" means "flips". So incrementing is
a cascade of controlled NOTs:

| bit | flips when | gate |
|---|---|---|
| `reg[0]` (MSB, worth 32) | all five lower bits are 1 | X on `reg[0]`, controlled on `reg[1..5]` |
| `reg[1]` (worth 16) | the four lower bits are 1 | X on `reg[1]`, controlled on `reg[2..5]` |
| $\vdots$ | $\vdots$ | $\vdots$ |
| `reg[5]` (LSB, worth 1) | always | plain X |

Two details make the loop correct.

**Order.** Walk from the most significant bit *down*. Each flip reads only bits below it,
and those have not been touched yet, so every control sees the register's *old* value —
which is what "the carry out of the low digits" means. Going the other way would read
already-updated bits and compute nonsense.

**The last line is not a special case.** The least significant bit flips on *no* controls,
and `with qc.control():` — a control scope over an empty list of controls — says precisely
that: "do this wherever all zero of these qubits are $\lvert 1\rangle$", which is
everywhere. The odometer's bottom digit really does have an empty carry condition.

A note on what this costs. `qsim` implements a multiply-controlled gate by **slicing the
control axes** of the state tensor: fix those axes to 1, apply the 2×2 matrix to the
surviving slab, leave the rest alone. Adding a control is free — no bigger matrix is ever
built (see the `apply_controlled` kernel and **[04 — combinators](../04-combinators.ipynb)**).
Real hardware has no such luxury: a 5-controlled X must be decomposed into one- and
two-qubit gates, dozens of them, usually with ancillas. The slice here is the mathematical
meaning those decompositions are working to implement, not a shortcut around them.

In [ ]:
@qsim.gate
def increment(qc: Circuit, reg: Register) -> None:
    """Add 1 modulo 2**len(reg) to an MSB-first register — a binary odometer.

    Bit i flips exactly when every bit below it is 1 (the carry has rippled all the
    way up to it). Working from the most significant bit downwards means each flip
    reads only bits that have not been updated yet, i.e. the register's old value.
    """
    for i in range(len(reg)):
        # reg[i + 1:] is every *less* significant bit. For the last bit that slice is
        # empty, and a control scope with no controls is an unconditional gate — the
        # odometer's bottom digit, which flips every single time.
        with qc.control(*reg[i + 1:]):
            X(reg[i])


# Undoing "+1" is "-1". The block algebra hands that over with no arithmetic at all:
# .adjoint() reverses the recorded ops and inverts each one (X is its own inverse, so
# only the *order* actually changes), and what comes back is a first-class Block.
decrement = increment.adjoint()

print(increment, "->", decrement)

`decrement` is not a function that happens to undo `increment`; it is a `Block`, with a
name, that `block_counts()` will report, and that can itself be controlled or inverted
again. Reversing a subroutine is an operation on subroutines, and the algebra is closed —
which is the whole point of Phase 2.75's block model.

It is worth pausing on how little we had to know. Nobody wrote a subtractor. Nobody worked
out that subtracting 1 borrows where incrementing carries. The *reason* `increment.adjoint()`
is a decrementer is that every gate is invertible, so every circuit is, so "run it
backwards" is always available and always means "the inverse map" — and the inverse of
$x \mapsto x + 1 \bmod 2^n$ is $x \mapsto x - 1 \bmod 2^n$.

Let us check it on all 64 basis states, forwards and backwards.

In [ ]:
increment_errors = 0
decrement_errors = 0
for value in range(N_SITES):
    up = Circuit(name="odometer", seed=0)
    reg_up = up.register(N_POS, name="x")
    reg_up.encode(value)                      # prepare the basis state |value>
    increment(up, reg_up)
    # A basis state has one amplitude of modulus 1; argmax over the flat probability
    # vector reads off which basis state that is, as an integer (qubit 0 = MSB).
    increment_errors += int(np.argmax(up.inspect.probabilities()) != (value + 1) % N_SITES)

    down = Circuit(name="odometer", seed=0)
    reg_down = down.register(N_POS, name="x")
    reg_down.encode(value)
    decrement(down, reg_down)
    decrement_errors += int(np.argmax(down.inspect.probabilities()) != (value - 1) % N_SITES)

print(f"increment wrong on {increment_errors} of {N_SITES} basis states")
print(f"decrement wrong on {decrement_errors} of {N_SITES} basis states")

# What one increment costs, on a register that has had nothing else done to it.
priced = Circuit(name="price", seed=0)
increment(priced, priced.register(N_POS, name="x"))
print("one increment:", priced.gate_counts(), priced.block_counts())
assert increment_errors == 0 and decrement_errors == 0

Six X gates, and the wrap-around ($63 + 1 = 0$, $0 - 1 = 63$) comes for free: dropping the
carry off the top of a 6-bit register *is* arithmetic modulo 64. The cycle is not something
we imposed on the walk; it is what a finite register does.

### 1b. The shift: one block, both directions

Now condition the odometer on the coin.

The $\lvert 1\rangle$ branch is direct — `with qc.control(coin):` wraps the block and every
op inside it gains `coin` as an extra control.

The $\lvert 0\rangle$ branch needs "run this where the control is **zero**", which `qsim`
(like real hardware) does not offer as a primitive. The standard trick is a sandwich:

$$X_c \,\cdot\, C_c[\,U\,] \,\cdot\, X_c$$

Flip the coin, so the branch we care about is now labelled $\lvert 1\rangle$; do the
controlled thing; flip it back. Reading it from the inside out: in the branch where the
coin was $\lvert 0\rangle$, the first $X$ makes it $\lvert 1\rangle$, the control fires,
and the second $X$ restores it. In the branch where the coin was $\lvert 1\rangle$, the
first $X$ makes it $\lvert 0\rangle$, the control does not fire, and the second $X$ puts it
back. Nothing about the coin is disturbed either way, which matters enormously — the coin
has to survive intact to be re-tossed next step.

`with qsim.within(X, coin):` is that sandwich as one construct: it applies $X$ now and
$X^\dagger$ on the way out, so the two halves cannot drift apart. (See
**[04 — combinators](../04-combinators.ipynb)**; `qsim.decoherence.pointer_coupling` is
built the same way.)

So the shift is four lines, and it reads as its own description: *right where the coin is
one, left where the coin is zero.*

In [ ]:
def shift(qc: Circuit, pos: Register, coin) -> None:
    """Position += 1 where the coin is |1>, position -= 1 where the coin is |0>."""
    with qc.control(coin):                 # the |1> branch: step right
        increment(qc, pos)
    with qsim.within(X, coin):             # X ... X turns "control on 1" into
        with qc.control(coin):             # "control on 0": the |0> branch steps left
            decrement(qc, pos)


# A quick sanity check on one definite coin value: coin |0> must move the walker left.
check = Circuit(name="shift-check", seed=0)
check_pos = check.register(N_POS, name="x")
check_coin = check.alloc("coin")
check_pos.encode(CENTER)
shift(check, check_pos, check_coin)
print("coin |0> sends site", CENTER, "->", int(np.argmax(check.inspect.probabilities())) // 2)
X(check_coin)                              # now the coin is |1>
shift(check, check_pos, check_coin)
shift(check, check_pos, check_coin)
print("two more steps with coin |1>:", int(np.argmax(check.inspect.probabilities())) // 2)

(The `// 2` divides out the coin, which is the least significant axis of the flattened
state: with the coin at the end of the register list, basis index $= 2\cdot\text{site} +
\text{coin}$.)

### 1c. The initial state, and why the coin is prepared *twice*

The walker starts at the centre site, `pos.encode(CENTER)` — plain X gates on a register
known to be $\lvert 0\ldots0\rangle$.

The coin is prepared in
$$\lvert \circlearrowleft \rangle = \frac{\lvert 0\rangle + i\lvert 1\rangle}{\sqrt2}$$
by $H$ then $S$ ($S$ multiplies the $\lvert 1\rangle$ amplitude by $i$). This is a
deliberate choice and it is worth a paragraph, because the obvious alternative — start the
coin in $\lvert 0\rangle$ — famously does something else.

The Hadamard coin is **not** left–right symmetric. $H$ treats its two inputs differently:
$H\lvert 0\rangle = \tfrac{1}{\sqrt2}(\lvert 0\rangle + \lvert 1\rangle)$ but
$H\lvert 1\rangle = \tfrac{1}{\sqrt2}(\lvert 0\rangle - \lvert 1\rangle)$, and that minus
sign biases the interference. Start from $\lvert 0\rangle$ and the walker **drifts**:
$\langle x \rangle \to -(1 - 1/\sqrt2)\,t \approx -0.293\,t$. (Part 6a shows this happening.)

A drifting distribution has a width *and* a moving centre, and $\sigma$ would then be
measuring a mixture of the two. The state $(\lvert 0\rangle + i \lvert 1\rangle)/\sqrt2$
is the fixed point of that asymmetry: the factor of $i$ exactly compensates $H$'s minus
sign, the distribution stays symmetric about the start for every $t$ and every $\theta$,
and $\langle x\rangle = 0$ to $10^{-15}$ throughout. Then $\sigma$ measures *spreading* and
nothing else — which is the only quantity this notebook is about.

### 1d. The environment: one fresh qubit per step, kept forever

`qc.environment_qubit()` allocates a qubit and marks it as "the rest of the world". The marking
changes no physics whatsoever: the qubit stays in the state tensor, it stays entangled with
everything it touched, and the global state stays exactly pure to the last step. Nothing is
traced out, nothing is measured, nothing stochastic happens anywhere in this notebook.

That is what makes the $\theta = \pi$ result *exact* rather than approximate. A simulator
that modelled decoherence by measuring the coin and averaging over outcomes would give the
same numbers to within sampling error; here there is no sampling error, because there is no
sampling. The price is one qubit per time step, and it is the reason this notebook spends
its qubit budget on **time** rather than on space (Part 2).

**Fresh** matters too. Each step gets a *new* environment qubit. Reusing one would let the
second coupling partly un-learn the first — the eraser of
**[quantum_eraser](quantum_eraser.ipynb)** by accident — and the walk would not be
memoryless.

In [ ]:
def position_marginal(qc: Circuit) -> np.ndarray:
    """P(site): the full outcome distribution, with coin and environment summed away."""
    # probabilities() is a flat vector of length 2**n indexed by the whole bit pattern.
    # Reshaping it to (2,)*n undoes that flattening and restores one axis per qubit, in
    # allocation order: axes 0..N_POS-1 are the position register, axis N_POS is the
    # coin, and every axis after that is one step's environment qubit.
    joint = qc.inspect.probabilities().reshape((2,) * qc.n_qubits)
    # Summing a joint distribution over a set of axes *is* the marginal over the rest:
    # "I do not care what those variables did." This is the honest measurement — the
    # same number a laboratory would get by measuring the position register and
    # ignoring everything else — not a peek at some privileged sub-state.
    marginal = joint.sum(axis=tuple(range(N_POS, qc.n_qubits)))
    # Flatten the position axes back to a single site index. C-order reshape walks the
    # last axis fastest, so axis 0 varies slowest: that *is* "qubit 0 is the MSB".
    return marginal.reshape(-1)


def spread(marginal: np.ndarray) -> tuple[float, float]:
    """Mean and standard deviation of the signed position, in sites."""
    mean = float(marginal @ SITES)                       # sum_x x P(x)
    variance = float(marginal @ SITES**2) - mean**2      # <x^2> - <x>^2
    return mean, float(np.sqrt(max(variance, 0.0)))


def walk(theta: float, *, symmetric: bool = True, t_steps: int = T_STEPS) -> dict:
    """Run the coined walk with coin-dephasing strength theta; record it step by step."""
    qc = Circuit(name=f"walk(theta={theta:.3f})", seed=2026)
    pos = qc.register(N_POS, name="x")
    coin = qc.alloc("coin")
    pos.encode(CENTER)
    if symmetric:
        H(coin)
        S(coin)                                     # (|0> + i|1>)/sqrt(2)

    marginals, coherences = [], []
    for _ in range(t_steps):
        H(coin)                                     # 1. toss
        env = qc.environment_qubit()                     # 2. a fresh piece of the world...
        dephasing_coupling(coin, env, theta=theta)   # ... looks at the coin
        coherences.append(qc.inspect.coherence(coin))
        shift(qc, pos, coin)                        # 3. left/right, in superposition
        marginals.append(position_marginal(qc))

    marginals = np.array(marginals)
    means, sigmas = zip(*(spread(m) for m in marginals))
    return {
        "circuit": qc,
        "marginals": marginals,                     # shape (t_steps, N_SITES)
        "mean": np.array(means),
        "sigma": np.array(sigmas),
        "coherence": np.array(coherences),
        "n_qubits": qc.n_qubits,
        "norm": qc.inspect.norm(),
        "edge_weight": float(marginals[:, [0, -1]].max()),
    }

## Part 2 — Seven walks

### Sizing the experiment

The register sizes are a deliberate trade, and the trade is **time against space**.

Every time step costs one environment qubit, and every position qubit doubles the site
count. With $T = 12$ steps the walker can be at most 12 sites from where it started, so a
64-site cycle is already more than twice the room it can possibly use — spending qubits on
a larger ring would buy nothing but empty sites, while spending them on more steps buys
more of the only axis the figures actually plot. Hence $n = 6$ position qubits (not the 8
one might reach for first), a coin, and 12 environment qubits: **19 qubits** at the end of
each run, $2^{19} = 524{,}288$ amplitudes.

There is one thing the extra room does buy, and it is not cosmetic. A cycle *wraps*: if any
amplitude reached site 0 or site 63 it would come round the other side and start
interfering with itself, and $\sigma$ — computed from signed positions — would be measuring
an artefact. So we assert at every recorded step that the two edge sites carry weight below
$10^{-12}$. They in fact carry exactly zero, because 12 steps cannot reach 32 sites, but an
assertion that has to be *checked* rather than argued is the one that survives a later edit
of `T_STEPS`.

The seven $\theta$ values are the dial: $0$, $\pi/8$, $\pi/4$, $3\pi/8$, $\pi/2$, $3\pi/4$,
$\pi$ — both endpoints and five points in between.

In [ ]:
runs = {float(theta): walk(float(theta)) for theta in THETAS}

print(f"{'theta/pi':>9} {'qubits':>7} {'sigma(T)':>9} {'max|mean|':>11} "
      f"{'edge weight':>12} {'|norm-1|':>10}")
for theta in THETAS:
    r = runs[float(theta)]
    print(f"{theta / np.pi:9.3f} {r['n_qubits']:7d} {r['sigma'][-1]:9.3f} "
          f"{np.abs(r['mean']).max():11.2e} {r['edge_weight']:12.2e} "
          f"{abs(r['norm'] - 1):10.2e}")

    # No wrap-around: nothing has reached the far side of the ring.
    assert r["edge_weight"] < 1e-12
    # The symmetric coin keeps the distribution centred, so sigma measures spreading only.
    assert np.abs(r["mean"]).max() < 1e-9
    # Still a unit vector: the environment is all still here, nothing was traced away.
    assert abs(r["norm"] - 1) < 1e-12

Read the last column first. After 12 steps of coupling a coin to a dozen environment
qubits, the global state is still a unit vector to $10^{-15}$ — still pure, still exactly
invertible. **Decoherence did not happen to the state.** It happened to the *marginal*, the
view in the second column of `position_marginal`, where the environment axes get summed
away. Put them back and every bit of interference is still there, byte for byte; that is
the point **[quantum_eraser](quantum_eraser.ipynb)** makes at length.

And $\sigma$ after 12 steps falls from 6.5 sites to 3.46 — the latter being exactly
$\sqrt{12}$, which Part 4 will show is no coincidence.

### The shape of the walker

Before the exponents, look at what the walker actually looks like. One parity remark first:
after $t$ steps the walker has taken $t$ steps of $\pm 1$, so its position has the same
parity as $t$. Half the sites are empty at any moment, by arithmetic and not by physics, so
the plots show only the occupied sublattice.

In [ ]:
occupied = (np.abs(SITES) <= T_STEPS) & (SITES % 2 == T_STEPS % 2)
x_occ = SITES[occupied]

# The exact simple-random-walk distribution after T steps, for comparison:
# P(x) = C(T, k) / 2**T with x = T - 2k, i.e. k left-steps out of T.
binomial = np.array([comb(T_STEPS, (T_STEPS - x) // 2) / 2**T_STEPS for x in x_occ])

fig_profile, ax_profile = plt.subplots(figsize=(9.0, 4.2))
for theta, color, label in [(0.0, "#c33b53", r"$\theta = 0$  (nothing is watching)"),
                            (np.pi / 2, "#b07d2b", r"$\theta = \pi/2$  (a partial record)"),
                            (np.pi, "#17797c", r"$\theta = \pi$  (a perfect record)")]:
    ax_profile.plot(x_occ, runs[float(theta)]["marginals"][-1][occupied],
                    "-o", ms=4.5, lw=1.8, color=color, label=label)
ax_profile.plot(x_occ, binomial, "k--", lw=1.3,
                label=r"exact binomial $\binom{12}{k}/2^{12}$")
ax_profile.axvline(T_STEPS / np.sqrt(2), color="#8a8f98", ls=":", lw=1.1)
ax_profile.axvline(-T_STEPS / np.sqrt(2), color="#8a8f98", ls=":", lw=1.1)
ax_profile.annotate(r"$\pm t/\sqrt{2}$", (T_STEPS / np.sqrt(2), 0.245), fontsize=9,
                    color="#555", textcoords="offset points", xytext=(5, 0))
ax_profile.set_xlabel("position $x$ (sites from the start)")
ax_profile.set_ylabel("probability after 12 steps")
ax_profile.set_ylim(0.0, 0.36)
ax_profile.set_title("the same walk, seen through three settings of one knob")
ax_profile.legend(fontsize=9, loc="upper center", ncol=2, framealpha=0.95)
fig_profile.tight_layout()

Three completely different distributions out of one circuit.

**$\theta = 0$ — the horns.** The undisturbed quantum walk piles its probability up at two
*wavefronts* near $\pm t/\sqrt2$ and leaves the middle almost flat. This is the signature
picture of the coined quantum walk and it is pure interference. Every site can be reached
by many different left/right histories; in the middle of the distribution those histories
arrive with a wide spread of phases and largely cancel, while near the extreme edge there
are very few histories and they arrive in phase, so their amplitudes *add* before being
squared. The result is weight pushed outward to the fastest-moving components — the
walker's mass ends up near the light-cone edge rather than under a bell curve. Note also
that it stops dead at $\pm 12$: nothing travels faster than one site per step, in this
theory as in any other.

**$\theta = \pi$ — the bell curve.** With the coin fully recorded every step, the horns are
gone and what is left is the binomial distribution, sitting exactly on the dashed line. The
central limit theorem, arrived at from a unitary circuit.

**$\theta = \pi/2$ — in between.** Vestigial shoulders where the horns used to be, filling
in toward the middle. No sharp transition anywhere; there is no point on the dial where the
walk stops being quantum.

## Part 3 — Why the exponent is the whole story

### The Khintchine / central-limit fact

Take $t$ steps $s_1, \ldots, s_t$, each $\pm1$, and let $x = \sum_i s_i$.

**Classically**, the steps are random and independent. Means add and *variances* add:

$$\operatorname{Var}(x) = \sum_i \operatorname{Var}(s_i) = t
\qquad\Longrightarrow\qquad \sigma(t) = \sqrt{t}.$$

The cross terms $\langle s_i s_j\rangle$ vanish for $i \ne j$ because the steps are
uncorrelated, and that vanishing is the whole reason for the square root. Random signs add
**in quadrature** — this is Khintchine's inequality in its familiar form, the same fact
that makes a random walk of $t$ steps end up $\sqrt t$ away and the standard error of a
mean fall like $1/\sqrt{n}$.

**Quantum-mechanically**, the steps are not random and there are no "the" steps. Every
left/right history is an amplitude, all histories are present at once, and the amplitudes
**add linearly** — and only after all the adding is done does the Born rule square the
result. Interference makes the effective step-to-step correlation $\langle s_i s_j\rangle$
*not* vanish: consecutive steps of an undisturbed coined walk are strongly correlated,
because the coin that steered step $i$ is still coherent when it steers step $i+1$. Sum
$t^2$ non-vanishing cross terms instead of $t$ diagonal ones and you get

$$\operatorname{Var}(x) \sim t^2 \qquad\Longrightarrow\qquad \sigma(t) \sim t.$$

So the exponent is a direct readout of *which norm the steps are adding in*: $\tfrac12$ for
probabilities in the 1-norm, $1$ for amplitudes in the 2-norm. That is the entire content
of **[l1_vs_l2](l1_vs_l2.ipynb)**, turned from a statement about matrices into a statement
about how far something gets.

And $\theta$ is the knob on the cross terms. Each dephasing multiplies the coin's
off-diagonal by $\cos(\theta/2)$, so the correlation between step $i$ and step $j$ is
suppressed by roughly $\cos(\theta/2)^{|i-j|}$: at $\theta = 0$ nothing is suppressed and
all $t^2$ terms survive; at $\theta = \pi$ every off-diagonal term dies immediately and
only the $t$ diagonal ones are left.

### The money figure

A power law $\sigma \sim t^{a}$ is a straight line of slope $a$ on log–log axes, so the
exponent can simply be read off — and fitted. We fit over $t \in [3, 12]$, dropping the
first two steps where a walk this short is still visibly discrete.

In [ ]:
def fit_exponent(sigma: np.ndarray, lo: int = FIT_FROM, hi: int = T_STEPS) -> float:
    """Least-squares slope of log(sigma) against log(t) over the window [lo, hi]."""
    window = (STEPS >= lo) & (STEPS <= hi)
    # np.polyfit(x, y, 1) returns [slope, intercept] of the least-squares straight line.
    # On log-log axes that slope *is* the exponent of the power law sigma ~ t**slope.
    return float(np.polyfit(np.log(STEPS[window]), np.log(sigma[window]), 1)[0])


exponents = np.array([fit_exponent(runs[float(theta)]["sigma"]) for theta in THETAS])
for theta, a in zip(THETAS, exponents):
    print(f"theta = {theta / np.pi:5.3f} pi   fitted exponent = {a:.4f}")

# The dial only ever turns one way: more watching cannot mean faster spreading.
assert np.all(np.diff(exponents) <= 1e-6)
# The two ends are the two textbook answers.
assert 0.85 <= exponents[0] <= 1.05
assert abs(exponents[-1] - 0.5) < 0.03
assert np.all(np.diff(runs[0.0]["sigma"]) > 0)

In [ ]:
fig_money, (ax_ll, ax_exp) = plt.subplots(1, 2, figsize=(11.5, 4.4))
colors = plt.get_cmap("plasma")(np.linspace(0.05, 0.85, len(THETAS)))

for theta, label, color in zip(THETAS, THETA_LABELS, colors):
    ax_ll.loglog(STEPS, runs[float(theta)]["sigma"], "-o", ms=3.6, lw=1.7, color=color,
                 label=rf"$\theta = ${label}")
# Guide lines, anchored at t = 3 so they sit among the data rather than off the axes.
ax_ll.loglog(STEPS, runs[0.0]["sigma"][2] * STEPS / 3, "k--", lw=1.1)
ax_ll.loglog(STEPS, runs[float(np.pi)]["sigma"][2] * np.sqrt(STEPS / 3), "k:", lw=1.3)
ax_ll.annotate(r"$\sigma \propto t$", (1.05, 0.70), fontsize=10)
ax_ll.annotate(r"$\sigma \propto t^{1/2}$", (7.6, 2.35), fontsize=10)
ax_ll.set_xlabel("steps $t$")
ax_ll.set_ylabel(r"width $\sigma(t)$  (sites)")
ax_ll.set_title("spreading, on log–log axes")
ax_ll.legend(fontsize=8, loc="upper left")

ax_exp.plot(THETAS / np.pi, exponents, "-o", color="#c33b53", lw=2.0, ms=6)
ax_exp.axhline(1.0, color="#8a8f98", ls="--", lw=1.1)
ax_exp.axhline(0.5, color="#8a8f98", ls=":", lw=1.3)
ax_exp.annotate("ballistic: reach $N$ in $\\sim N$ steps", (0.02, 1.005), fontsize=9)
ax_exp.annotate("diffusive: reach $N$ in $\\sim N^2$ steps", (0.02, 0.455), fontsize=9)
ax_exp.set_xlabel(r"$\theta / \pi$ — how hard the world watches the coin")
ax_exp.set_ylabel(r"fitted exponent over $t \in [3, 12]$")
ax_exp.set_ylim(0.42, 1.08)
ax_exp.set_title("one knob, two textbook exponents")
fig_money.tight_layout()

The left panel is seven straight-ish lines fanning out between two slopes; the right panel
is their slopes. The dial runs from $0.96$ at $\theta = 0$ to exactly $0.500$ at
$\theta = \pi$, passing continuously through everything in between.

**The $N$-versus-$N^2$ reading.** Turn the exponent around and ask how many steps it takes
to reach distance $N$: ballistic transport needs $\sim N$, diffusion needs $\sim N^2$. That
single square is where quantum-walk algorithms get their speedups — a quantum walk explores
a graph quadratically faster than the classical random walk on the same graph, and
Grover's search is exactly that statement on the complete graph ($\sqrt{N}$ queries instead
of $N$). We will build Grover in Phase 6; this figure is the reason it works, drawn before
the algorithm exists. And the figure also says what it costs: the speedup lives in the
cross terms, and the cross terms are what an eavesdropping environment destroys first.

**Two honest wrinkles in the left panel.** The $\theta = 0$ curve is very slightly
*convex* — its fitted slope $0.96$ is still climbing toward $1$, because a 12-step walk has
not finished its transient. And no curve is perfectly straight, because we are fitting a
power law over less than one decade. Neither wrinkle threatens the claim, which is about
the *ordering* of the exponents and their two limits, but both are reasons to read Part 5
before quoting any middle number.

## Part 4 — The $\theta = \pi$ endpoint, exactly

The right-hand end of the dial is not "approximately classical". It is classical, on the
nose, and the argument is short enough to check line by line.

**Step 1: after a perfect record, the coin is a fair classical bit.**
`dephasing_coupling(coin, env, theta=pi)` rotates the environment qubit by $\pi$ exactly
where the coin is $\lvert 1\rangle$, so the environment ends in $\lvert 0_E\rangle$ or
$\lvert 1_E\rangle$ according to the coin — orthogonal states, a perfect record. The coin's
own reduced density matrix loses its off-diagonal entirely and becomes
$\rho = \tfrac12 I$: probabilities $(\tfrac12, \tfrac12)$, coherence $0$.

**Step 2: the maximally mixed coin is a fixed point of the toss.** $H \cdot \tfrac12 I \cdot
H^\dagger = \tfrac12 H H^\dagger = \tfrac12 I$. So the *next* toss starts from a fair coin
too, and the next, forever. Each step therefore contributes a fair $\pm1$ sign,
independent of every other step — the record of step $i$ lives in environment qubit $i$ and
nothing later touches it.

**Step 3: that is the definition of simple random walk.** So
$\operatorname{Var}(x) = t$ exactly, and $P(x)$ is exactly the binomial.

**And this is E0 Part 5, arrived at from the other side.** There the bridge was:
$S_{ij} = \lvert U_{ij}\rvert^2$ is doubly stochastic, and full dephasing after every gate
turns the quantum simulator into the classical one with matrix $S$. Here $U = H$, so

$$S = \lvert H \rvert^2 = \begin{pmatrix} \tfrac12 & \tfrac12 \\[2pt] \tfrac12 & \tfrac12
\end{pmatrix} = \texttt{COIN},$$

the fair-coin-flip matrix from E0 Part 0 — the rank-1 projector that throws away its input
and returns the uniform distribution. The fully dephased Hadamard coin is not *like* a
coin flip. It *is* `COIN`, and the walk it drives is *the* random walk, with the binomial
coefficients correct to the last bit of the mantissa.

In [ ]:
H_MATRIX = np.array([[1, 1], [1, -1]]) / np.sqrt(2)
COIN = np.abs(H_MATRIX) ** 2                  # |H_ij|^2, entry by entry -- E0's COIN
print("|H|^2 =\n", COIN)

pi_run = runs[float(np.pi)]
pi_circuit = pi_run["circuit"]

# The coin's reduced state was recorded right after each dephasing; check every one.
worst_mixed = float(np.abs(pi_run["coherence"]).max())   # |rho_01| should be 0 each time
final_coin_rho = pi_circuit.inspect.reduced_density_matrix([pi_circuit.qubits[N_POS]])

# sigma^2(t) == t, exactly, at every step.
worst_variance = float(np.abs(pi_run["sigma"] ** 2 - STEPS).max())

# And the whole distribution, not just its second moment: P(x) = C(t, k)/2^t.
worst_binomial = 0.0
for t, marginal in zip(STEPS, pi_run["marginals"]):
    exact = np.zeros(N_SITES)
    for k in range(t + 1):                      # k left-steps out of t
        exact[CENTER + t - 2 * k] = comb(t, k) / 2**t
    worst_binomial = max(worst_binomial, float(np.abs(marginal - exact).max()))

print(f"\ncoin coherence |rho_01| after each dephased toss, worst: {worst_mixed:.2e}")
print(f"coin reduced density matrix at the end:\n{final_coin_rho.real}")
print(f"worst |sigma^2(t) - t| over 12 steps:        {worst_variance:.2e}")
print(f"worst |P(x) - binomial(t, x)| over 12 steps: {worst_binomial:.2e}")

assert np.allclose(COIN, 0.5, atol=1e-15)
assert np.abs(final_coin_rho - np.eye(2) / 2).max() < 1e-12
assert worst_mixed < 1e-12
assert worst_variance < 1e-9
assert worst_binomial < 1e-12

$\sigma^2(t) = t$ to a few times $10^{-14}$, and the *entire distribution* agrees with
$\binom{12}{k}/2^{12}$ to $4\times10^{-16}$ — which is to say, to the last bit of a double.
This is the strongest claim in the notebook and it is worth being clear about what makes it
strong: **the classical limit here is not a model, an average, or a limit taken by hand.**
It is a 19-qubit unitary circuit, run once, with nothing measured and nothing discarded,
and the binomial coefficients fall out of it at machine precision.

Two things had to be true for that, and both are structural rather than lucky. The
environment qubits are **fresh** each step, so the records are independent and the coin
cannot recover what it lost. And they are **kept** — never traced out, never measured — so
there is no sampling error anywhere to hide behind.

## Part 5 — The honesty note: what the sliding exponent is not

The right panel of the money figure is easy to over-read, so here is exactly what it does
and does not say.

**It is not a claim of intermediate asymptotics.** For *any* $\theta > 0$ — however small —
this walk is **asymptotically diffusive**. The true long-time exponent is $\tfrac12$ the
moment the environment learns anything at all, because each step's record is independent
and permanent, so after enough steps the coin's memory of its own past is gone whatever the
per-step suppression $\cos(\theta/2)$ was. There is no $\theta$ at which the walk spreads
like $t^{0.83}$ forever. The exponent $0.83$ is not a physical exponent; it is a number our
fitting window returned.

What actually happens is a **crossover**. A decohered quantum walk is ballistic for
$t \lesssim t_c$ and diffusive for $t \gtrsim t_c$, where the crossover time grows as the
coupling weakens — roughly $t_c \sim 1/\theta^2$ for small $\theta$, since that is how many
steps it takes for the accumulated suppression $\cos(\theta/2)^t$ to matter. Our window is
fixed at $t \in [3, 12]$. So the right panel is a picture of **where the fixed window sits
relative to a moving crossover**: at large $\theta$ the crossover is behind us and we
measure $\tfrac12$; at small $\theta$ it is far ahead and we measure nearly $1$; in between
we straddle it and get an average of the two slopes.

That is a real, reproducible, physically meaningful measurement — it is what any finite
experiment reports — but it is a statement about a window, not about $t \to \infty$.

**The data can say this itself.** Split the fitting window in half and fit each piece. If a
$\theta$ were genuinely a power law with exponent $a$, both halves would return $a$. If the
walk is crossing over inside our window, the late half must come back *lower* than the early
half.

In [ ]:
early = np.array([fit_exponent(runs[float(t)]["sigma"], 3, 7) for t in THETAS])
late = np.array([fit_exponent(runs[float(t)]["sigma"], 7, 12) for t in THETAS])

print(f"{'theta/pi':>9} {'t in [3,7]':>11} {'t in [7,12]':>12} {'drift':>8}")
for theta, a_early, a_late in zip(THETAS, early, late):
    print(f"{theta / np.pi:9.3f} {a_early:11.4f} {a_late:12.4f} {a_late - a_early:+8.4f}")

fig_window, ax_window = plt.subplots(figsize=(7.2, 4.2))
ax_window.plot(THETAS / np.pi, early, "-o", color="#b07d2b", lw=1.8, ms=5,
               label=r"early window $t \in [3, 7]$")
ax_window.plot(THETAS / np.pi, late, "-s", color="#17797c", lw=1.8, ms=5,
               label=r"late window $t \in [7, 12]$")
ax_window.plot(THETAS / np.pi, exponents, "--", color="#c33b53", lw=1.4,
               label=r"full window $t \in [3, 12]$")
ax_window.axhline(0.5, color="#8a8f98", ls=":", lw=1.2)
ax_window.set_xlabel(r"$\theta / \pi$")
ax_window.set_ylabel("fitted exponent")
ax_window.set_title("the exponent is a property of the window, not of the walk")
ax_window.legend(fontsize=9)
fig_window.tight_layout()

# An exact power law does not care which window you fit it over.
assert abs(early[-1] - 0.5) < 1e-9 and abs(late[-1] - 0.5) < 1e-9
# At theta = pi/2 the crossover is happening inside our twelve steps: the walk is
# measurably slower over the second half of the window than over the first.
assert late[4] < early[4] - 1e-3
# At theta = pi/4 it is not: the walk is still speeding up toward its ballistic limit.
assert late[2] > early[2]

Exactly the predicted signature, and it changes sign where it should.

For $\theta \ge 3\pi/8$ the late window comes back **below** the early one: the crossover is
happening inside our 12 steps, and the walk is already slowing toward $\tfrac12$ while we
watch. For $\theta \le \pi/4$ the late window comes back **above** the early one — the walk
is still *accelerating* toward its ballistic asymptote, because at that coupling $t_c$ is
far beyond step 12 and we are seeing only the pre-crossover regime plus the transient the
$\theta = 0$ curve has too. And at $\theta = \pi$ both windows return $0.5000$, because
there the power law is exact and windows do not matter.

So the sliding exponent is real and it is honest, provided you read it as: *how ballistic
does this walk look over the first dozen steps?* Which, incidentally, is the only question
a laboratory with a finite coherence time can ask either.

## Part 6a — Aside: the coin we did not use

Part 1c claimed the Hadamard walk started from a $\lvert 0\rangle$ coin drifts sideways.
It is a beloved fact and it costs one extra run to show. Same circuit, same $\theta = 0$,
only the coin preparation removed.

In [ ]:
asymmetric = walk(0.0, symmetric=False)
symmetric = runs[0.0]

print("mean position after each step")
print("  coin |0>              :", np.round(asymmetric["mean"], 3))
print("  coin (|0> + i|1>)/√2  :", np.abs(symmetric["mean"]).max(), "(max |mean|)")
print(f"\nasymptotic prediction -(1 - 1/√2)·t at t=12: {-(1 - 1 / np.sqrt(2)) * 12:.3f}")

fig_asym, (ax_drift, ax_shape) = plt.subplots(1, 2, figsize=(11.0, 3.9))
ax_drift.plot(STEPS, asymmetric["mean"], "-o", ms=4, color="#c33b53",
              label=r"coin $|0\rangle$")
ax_drift.plot(STEPS, symmetric["mean"], "-o", ms=4, color="#17797c",
              label=r"coin $(|0\rangle + i|1\rangle)/\sqrt{2}$")
ax_drift.plot(STEPS, -(1 - 1 / np.sqrt(2)) * STEPS, "k--", lw=1.1,
              label=r"$-(1 - 1/\sqrt{2})\,t$")
ax_drift.set_xlabel("steps $t$")
ax_drift.set_ylabel(r"mean position $\langle x \rangle$")
ax_drift.set_title("the Hadamard coin is not left–right symmetric")
ax_drift.legend(fontsize=9)

ax_shape.plot(x_occ, asymmetric["marginals"][-1][occupied], "-o", ms=4, color="#c33b53",
              label=r"coin $|0\rangle$")
ax_shape.plot(x_occ, symmetric["marginals"][-1][occupied], "-o", ms=4, color="#17797c",
              label=r"coin $(|0\rangle + i|1\rangle)/\sqrt{2}$")
ax_shape.set_xlabel("position $x$ after 12 steps")
ax_shape.set_ylabel("probability")
ax_shape.set_title("lopsided horns versus symmetric ones")
ax_shape.legend(fontsize=9)
fig_asym.tight_layout()

assert asymmetric["mean"][-1] < -2.0                     # it really does drift
assert np.abs(symmetric["mean"]).max() < 1e-9            # and the symmetric coin does not

The red walker slides left at a steady $0.293$ sites per step, tracking the analytic
prediction $-(1 - 1/\sqrt2)t$, and its two horns have unequal heights. Nothing about the
*shift* is asymmetric — `increment` and its adjoint are exact mirror images. The asymmetry
is entirely in the coin: $H\lvert 1\rangle$ carries a minus sign that $H\lvert 0\rangle$
does not, and that sign biases which histories interfere constructively.

The teal walker uses the $i$ to cancel exactly that bias. Both walkers are equally quantum
and spread at the same rate; only one of them has a $\sigma$ that means what we want it to
mean.

## Part 6b — Aside: the coin's own coherence

`inspect.coherence(coin)` is $\lvert \rho_{01}\rvert$ of the coin's reduced density matrix
— the single number **[decoherence_dial](decoherence_dial.ipynb)** puts on its $y$ axis, and
the exact quantity the dial there controls. We recorded it right after each dephasing, i.e.
at the moment the coin is about to steer a shift.

In [ ]:
fig_coh, ax_coh = plt.subplots(figsize=(7.6, 4.0))
for theta, label, color in [(0.0, r"$0$", "#c33b53"), (np.pi / 4, r"$\pi/4$", "#b07d2b"),
                           (np.pi / 2, r"$\pi/2$", "#17797c"), (np.pi, r"$\pi$", "#5b4b8a")]:
    ax_coh.plot(STEPS, runs[float(theta)]["coherence"], "-o", ms=4, lw=1.7, color=color,
                label=rf"$\theta = ${label}")
ax_coh.set_xlabel("steps $t$")
ax_coh.set_ylabel(r"coin coherence $|\rho_{01}|$")
ax_coh.set_title("the coin's coherence, measured just before each shift")
ax_coh.legend(fontsize=9)
fig_coh.tight_layout()

print("theta = 0   :", np.round(runs[0.0]["coherence"], 4))
print("theta = pi/2:", np.round(runs[float(np.pi / 2)]["coherence"], 4))
print("theta = pi  :", np.round(runs[float(np.pi)]["coherence"], 4))

Two features, and the second one is the more instructive.

**The envelope falls with $\theta$**, which is the expected reading: each dephasing
multiplies the off-diagonal by $\cos(\theta/2)$, and by $\theta = \pi$ the coin's coherence
is exactly zero from the first step onward — the flat purple line at the bottom, which is
Part 4's argument drawn.

**But at $\theta = 0$ the coherence is not constant either**, and it hits exactly zero at
step 2 even though *nothing is watching*. That is not decoherence in the sense this notebook
is about; it is the shift entangling the coin with the position register. After one step the
coin is perfectly correlated with a position it has moved to a different site — two
orthogonal branches — so the coin's *marginal* is maximally mixed. Look at the coin alone
and it appears to have lost everything. It has not: the walk is perfectly ballistic, and the
coherence comes back at step 3 as histories reconverge on the same sites and interfere.

Which is the whole moral of Track B in miniature. A reduced density matrix looking mixed
tells you the subsystem is entangled with something; it does not tell you the entanglement
is *irreversible*. The position register is part of the machine, and interference with it
resumes. The environment qubits are also still there and also still reversible — the
difference is only that we chose not to look at them, and that choice is what
`qc.environment()` marks. **Decoherence is a fact about your bookkeeping**, and the flat
purple line and the oscillating red one are the two ends of that sentence.

## Closing

One circuit, one knob, and both halves of a dichotomy that usually needs two chapters.

| | $\theta = 0$ | $\theta = \pi$ |
|---|---|---|
| what the world knows about the coin | nothing | everything, every step |
| how steps combine | amplitudes add, then get squared | probabilities add |
| step–step correlation | survives, $\sim t^2$ cross terms | zero, $t$ diagonal terms |
| $\sigma(t)$ | $\propto t$ (ballistic) | $= \sqrt{t}$ (diffusive) |
| final profile | two horns at $\pm t/\sqrt2$ | the binomial, exactly |
| steps to reach distance $N$ | $\sim N$ | $\sim N^2$ |
| the coin, as a matrix | $H$ | $\lvert H\rvert^2 = \texttt{COIN}$ |

The bottom row is the one to remember. Squaring $H$ entry by entry turns a unitary into a
stochastic matrix — that is E0's bridge, an algebraic fact about matrices. This notebook
shows the same map happening *dynamically*, to a machine nobody told about classical
physics: couple a coin to the world, and its $\lvert H_{ij}\rvert^2$ shows up in the
statistics of where a walker ends up, to fifteen decimal places.

And the global state was pure the entire time. The classical world was not created here. It
is what this one looks like from inside, by an observer who has declined to keep the
receipts.

### Where to go next

- **[l1_vs_l2](l1_vs_l2.ipynb)** (E0) Part 5 — the same bridge, proved on matrices rather
  than watched in a walk.
- **[decoherence_dial](decoherence_dial.ipynb)** (B1) — Part 6b's coherence, on a single
  qubit, against the $\cos(\theta/2)$ prediction.
- **[quantum_eraser](quantum_eraser.ipynb)** (B2) — the environment qubits we kept: run the
  couplings backwards and the horns come back.
- **[einselection](einselection.ipynb)** (B3) — why the *coin basis* is the one that gets
  recorded, and what a different coupling would classicalize instead.
- **[04 — combinators](../04-combinators.ipynb)** — `control`, `within`, and the block
  algebra that made `decrement` free.
- `grover_and_dj.ipynb` (Phase 6, not yet built) — the $N$-versus-$N^2$ gap, cashed in as
  an algorithm.

## Assertions

Every claim above, re-checked in one place, so the notebook fails loudly under
`jupyter execute` if any of it ever stops being true.

In [ ]:
# --- Part 1: the odometer, and its adjoint, on every basis state ---
assert increment_errors == 0, "increment is not +1 mod 2**n"
assert decrement_errors == 0, "increment.adjoint() is not -1 mod 2**n"

# --- Part 2: every run is clean, centred, unwrapped, and still a unit vector ---
for theta_check in THETAS:
    r = runs[float(theta_check)]
    assert r["n_qubits"] == N_POS + 1 + T_STEPS, "unexpected qubit count"
    # No amplitude reached the far side of the ring, so sigma is not measuring a wrap.
    assert r["edge_weight"] < 1e-12, "the walker wrapped around the cycle"
    # The symmetric coin keeps <x> = 0, so sigma measures spreading and not drift.
    assert np.abs(r["mean"]).max() < 1e-9, "the walk drifted"
    # Nothing was traced away: all 12 environment qubits are still in the state.
    assert abs(r["norm"] - 1) < 1e-12, "the global state is not a unit vector"

# --- Part 3: the dial slides one way, between the two textbook exponents ---
assert np.all(np.diff(exponents) <= 1e-6), "watching harder made the walk spread faster"
assert 0.85 <= exponents[0] <= 1.05, "the undisturbed walk is not ballistic"
assert abs(exponents[-1] - 0.5) < 0.03, "the fully dephased walk is not diffusive"
assert np.all(np.diff(runs[0.0]["sigma"]) > 0), "sigma(t) is not strictly increasing"

# --- Part 4: theta = pi is simple random walk, exactly ---
assert np.allclose(COIN, 0.5, atol=1e-15), "|H_ij|^2 is not the fair COIN matrix"
assert worst_mixed < 1e-12, "the coin was not maximally mixed after a perfect record"
assert np.abs(final_coin_rho - np.eye(2) / 2).max() < 1e-12, "coin rho != I/2"
assert worst_variance < 1e-9, "sigma^2(t) != t"
assert worst_binomial < 1e-12, "the position distribution is not the exact binomial"

# --- Part 5: the crossover is inside the window at large theta, ahead of it at small ---
assert abs(early[-1] - 0.5) < 1e-9, "an exact power law came out window-dependent"
assert abs(late[-1] - 0.5) < 1e-9, "an exact power law came out window-dependent"
assert late[4] < early[4] - 1e-3, "no crossover visible at theta = pi/2"
assert late[2] > early[2], "theta = pi/4 was already past its crossover"

# --- Part 6a: the |0> coin drifts, the symmetric one does not ---
assert asymmetric["mean"][-1] < -2.0, "the Hadamard walk from |0> did not drift"
assert np.abs(symmetric["mean"]).max() < 1e-9, "the symmetric coin drifted"

print("all assertions passed")